# TP — Voyage impossible (impossible travel)

## Objectif

Détecter les **déplacements impossibles** — un utilisateur qui se
connecte depuis deux lieux trop éloignés en trop peu de temps (vitesse
supérieure à **900 km/h**, la vitesse d'un avion de ligne) — puis éliminer les
faux positifs pour ne garder que le compte réellement compromis.

**Consignes :** complétez chaque fonction signalée par un encadré « ✏️ Exercice »,
puis exécutez la cellule de vérification qui la suit.

**Cadre légal — données simulées :** ce jeu de données est **entièrement
simulé** (adresses IP réservées à la documentation, RFC 5737). Toute analyse
de journaux d'authentification réels exige une **autorisation écrite**.
**Art. 323-1 Code pénal :** accès frauduleux = délit, même « juste pour voir ».


## Préambule : les données

Le SIEM vous remet un journal de connexions **minimal** : ni géolocalisation,
ni information sur les VPN. Ces informations se **reconstruisent** en joignant
deux autres fichiers, comme avec une vraie base GeoIP.

| fichier | colonnes | rôle |
| --- | --- | --- |
| `connexions.csv` | `timestamp`, `username`, `ip` | le journal brut des connexions |
| `ip_geo.csv` | `ip`, `latitude`, `longitude`, `city`, `country` | géolocalisation de chaque IP |
| `vpn_ips.csv` | `ip` | IP de sortie VPN/datacenter connues |

**Aide pandas :**
- `pd.read_csv(..., parse_dates=["timestamp"])` : charge et convertit la colonne en dates.
- `.head()`, `.info()` : premier aperçu.
- Doc : https://pandas.pydata.org/docs/user_guide/10min.html


In [ ]:
import pandas as pd
import numpy as np

connexions = pd.read_csv("connexions.csv", parse_dates=["timestamp"])
geo = pd.read_csv("ip_geo.csv")
vpn = pd.read_csv("vpn_ips.csv")

print(connexions.shape, geo.shape, vpn.shape)
connexions.head()



In [ ]:
connexions.info()
geo.head()



## 1. Détection des déplacements impossibles


### 1.1 Reconstruire la géolocalisation

`connexions.csv` ne contient que l'IP. Pour calculer des distances, il faut
ajouter la latitude/longitude de chaque IP via une **jointure** (`merge`) sur
la colonne commune `ip`.

**À propos de `merge` :** `pd.merge(gauche, droite, on="ip", how="left")`
joint deux DataFrames sur une colonne commune. `how="left"` garde toutes les
lignes de gauche, même si aucune géolocalisation n'existe (la latitude serait
alors `NaN`).
Doc : https://pandas.pydata.org/docs/reference/api/pandas.merge.html


<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Complétez `fusionner` : elle joint les connexions à la géolocalisation de leur IP
(colonnes ajoutées : `latitude`, `longitude`, `city`, `country`).

</fieldset>

In [ ]:
def fusionner(connexions: pd.DataFrame, geo: pd.DataFrame) -> pd.DataFrame:
    """Joint les connexions à la géolocalisation de leur IP.

    Colonnes ajoutées : latitude, longitude, city, country.
    """
    # BEGIN
    # raise NotImplementedError
    return pd.merge(connexions, geo, on="ip", how="left")
    # END



In [ ]:
df = fusionner(connexions, geo)
assert "latitude" in df.columns and "longitude" in df.columns
assert not df["latitude"].isna().any(), "des IP n'ont pas de géolocalisation"
print("ex 1 OK :", df.shape[0], "lignes,", df["ip"].nunique(), "IP distinctes")



### 1.2 Détection naïve : tri, précédent, vitesse

Pour détecter un déplacement impossible, on compare chaque connexion d'un
utilisateur à sa connexion **précédente** (même idée que `get_diffs` du TD
beacon, qui comparait chaque instant au précédent) :

1. **Trier** par `username` puis `timestamp` — sans ce tri, on comparerait
   deux connexions d'utilisateurs différents.
2. **Parcourir** les lignes en retenant la précédente (« shift à la main ») :
   si elle appartient au **même utilisateur**, on calcule la distance et la
   durée entre les deux ; sinon on l'ignore.
3. Calculer la **vitesse** = distance (haversine) / durée.
4. Alerter si la vitesse dépasse 900 km/h.

La distance du grand cercle est fournie ci-dessous (inutile de la réécrire) :


In [ ]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distance du grand cercle (km) entre deux points (lat/lon en degrés)."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))



<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Complétez `detecter_voyages_impossibles` : elle renvoie une ligne par connexion dont la vitesse depuis la connexion précédente du même utilisateur dépasse `seuil_kmh`, en suivant les étapes 1 à 4 ci-dessus.

</fieldset>

In [ ]:
def detecter_voyages_impossibles(df: pd.DataFrame, seuil_kmh: float = 900) -> pd.DataFrame:
    """Renvoie les connexions impliquant un déplacement > seuil_kmh.

    Colonnes retournées : username, timestamp, ip, t_prev, ip_prev,
    distance_km, duree_h, vitesse_kmh (une ligne par alerte).
    """
    # BEGIN
    # raise NotImplementedError
    ## Indice : trier par ["username", "timestamp"], puis parcourir les lignes
    ## en retenant la précédente (comme get_diffs du TD beacon).
    ## Indice : ne comparer que si la précédente est du même utilisateur.
    ## Indice : distance avec haversine_km, durée en heures avec .total_seconds() / 3600.
    df = df.sort_values(["username", "timestamp"])
    lignes = []
    prev = None
    for row in df.itertuples(index=False):
        if prev is not None and prev.username == row.username:
            distance = haversine_km(prev.latitude, prev.longitude, row.latitude, row.longitude)
            duree = (row.timestamp - prev.timestamp).total_seconds() / 3600
            vitesse = distance / duree if duree > 0 else float("inf")
            if distance > 0 and vitesse > seuil_kmh:
                lignes.append({
                    "username": row.username,
                    "timestamp": row.timestamp,
                    "ip": row.ip,
                    "t_prev": prev.timestamp,
                    "ip_prev": prev.ip,
                    "distance_km": distance,
                    "duree_h": duree,
                    "vitesse_kmh": vitesse,
                })
        prev = row
    colonnes = ["username", "timestamp", "ip", "t_prev", "ip_prev",
                "distance_km", "duree_h", "vitesse_kmh"]
    return pd.DataFrame(lignes, columns=colonnes)
    # END



In [ ]:
def check_detecter():
    # données à la main : (0, 0) -> (0, 90) en 1 h (~10 000 km) : alerte ;
    # (0, 0) -> (0, 1) en 10 h (~111 km) : pas d'alerte ; un seul point : rien.
    petit = pd.DataFrame({
        "timestamp": pd.to_datetime(["2024-01-01 10:00", "2024-01-01 11:00",
                                     "2024-01-01 10:00", "2024-01-01 20:00",
                                     "2024-01-01 12:00"]),
        "username": ["a", "a", "b", "b", "c"],
        "ip": ["1", "2", "1", "3", "1"],
        "latitude": [0.0, 0.0, 0.0, 0.0, 0.0],
        "longitude": [0.0, 90.0, 0.0, 1.0, 0.0],
    })
    res = detecter_voyages_impossibles(petit)
    assert len(res) == 1, f"attendu 1 alerte, obtenu {len(res)}"
    assert res.iloc[0]["username"] == "a"
    assert res.iloc[0]["ip_prev"] == "1" and res.iloc[0]["ip"] == "2"
    assert 10000 < res.iloc[0]["vitesse_kmh"] < 10100

    alertes = detecter_voyages_impossibles(df)
    assert len(alertes) == 120, f"attendu 120 alertes, obtenu {len(alertes)}"
    assert alertes["username"].nunique() == 18
    print("ex 2 OK")

check_detecter()


Après la vérification, la cellule suivante affiche les alertes : il y en a 120, mais la plupart sont des faux positifs.


In [ ]:
alertes = detecter_voyages_impossibles(df)
print(len(alertes), "alertes,", alertes["username"].nunique(), "utilisateurs distincts")
alertes

## 2. Éliminer les faux positifs


Le détecteur remonte **beaucoup** d'alertes. La plupart sont des **faux
positifs** : la position d'une IP n'est **pas** la position de l'utilisateur.

**Deux sources de faux positifs :**
1. **Le VPN** : un utilisateur chez lui se connecte via un nœud de sortie VPN
   à l'étranger → sa position semble « sauter » de sa ville vers le nœud.
2. **L'IP d'entreprise** : des employés sortent par un **même** proxy/NAT,
   géolocalisé au siège → chacun semble « sauter » de son domicile vers le
   siège.

On va les éliminer l'un après l'autre.


### 2.1 Éliminer les VPN (la liste)

Les nœuds de sortie VPN sont **connus** et listés dans `vpn_ips.csv`. On filtre
les connexions dont l'IP figure dans cette liste.

**Pourquoi une liste suffit ici :** un nœud de sortie VPN peut n'être utilisé
que par **un seul** de nos utilisateurs : il n'est pas « partagé », donc
indétectable par une simple statistique. Seule une **liste externe** (ou une
base d'ASN de datacenters) l'attrape.


<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3</legend>

Complétez `retirer_vpn` : elle renvoie les connexions dont l'IP n'est **pas** une sortie VPN connue.

</fieldset>

In [ ]:
def retirer_vpn(df: pd.DataFrame, vpn_ips: pd.DataFrame) -> pd.DataFrame:
    """Renvoie les connexions dont l'IP n'est PAS une sortie VPN connue."""
    # BEGIN
    # raise NotImplementedError
    return df[~df["ip"].isin(vpn_ips["ip"])]
    # END



In [ ]:
def check_retirer_vpn():
    petit = pd.DataFrame({"ip": ["1", "2", "3", "2"], "username": list("abcd")})
    res = retirer_vpn(petit, pd.DataFrame({"ip": ["2"]}))
    assert list(res["username"]) == ["a", "c"], "mauvaises lignes conservées"

    sans_vpn = retirer_vpn(df, vpn)
    alertes_vpn = detecter_voyages_impossibles(sans_vpn)
    assert len(alertes_vpn) == 85
    assert alertes_vpn["username"].nunique() == 13
    print("ex 3 OK : les télétravailleurs VPN ne sont plus alertés")

check_retirer_vpn()


La cellule suivante affiche le nombre d'alertes restantes une fois les VPN retirés.


In [ ]:
sans_vpn = retirer_vpn(df, vpn)
alertes_vpn = detecter_voyages_impossibles(sans_vpn)
print(len(alertes_vpn), "alertes,", alertes_vpn["username"].nunique(), "utilisateurs")

### 2.2 Éliminer l'IP d'entreprise (l'heuristique)

L'IP de sortie de l'entreprise n'est dans **aucune** liste. Mais elle est
**partagée** : de nombreux utilisateurs distincts se connectent derrière elle.

**Heuristique : une IP partagée = infrastructure.** `df.groupby("ip")["username"].nunique()`
compte les utilisateurs distincts par IP. Une IP résidentielle n'est vue que
par 1 (ou très peu) d'utilisateurs ; un proxy/NAT/VPN est vu par **beaucoup**.

En vrai, on fixe un seuil : une IP partagée par une poignée de membres d'une
même famille reste légitime. Ici, l'IP d'entreprise est partagée par
12 personnes, une IP résidentielle par 1.


<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Complétez `ip_partagees` : elle renvoie l'ensemble des IP vues par au moins `seuil` utilisateurs distincts.

</fieldset>

In [ ]:
def ip_partagees(df: pd.DataFrame, seuil: int = 2) -> set[str]:
    """IP vues par au moins `seuil` utilisateurs distincts (infra partagée)."""
    # BEGIN
    # raise NotImplementedError
    nb_users = df.groupby("ip")["username"].nunique()
    return set(nb_users[nb_users >= seuil].index)
    # END



In [ ]:
def check_ip_partagees():
    petit = pd.DataFrame({"ip": ["1", "1", "2", "2", "3"],
                          "username": ["a", "b", "a", "a", "c"]})
    assert ip_partagees(petit) == {"1"}, "seuil=2 : seule l'IP 1 a 2 utilisateurs"
    assert ip_partagees(petit, seuil=3) == set()

    partagees = ip_partagees(retirer_vpn(df, vpn))
    assert partagees == {"203.0.113.10"}, f"IP partagées : {partagees}"
    print("ex 4 OK")

check_ip_partagees()


La cellule suivante affiche le nombre d'utilisateurs derrière l'IP partagée.


In [ ]:
nb = sans_vpn[sans_vpn["ip"] == "203.0.113.10"]["username"].nunique()
print(ip_partagees(sans_vpn), "partagé par", nb, "utilisateurs")

### 2.3 Synthèse : qui est réellement compromis ?

On applique les deux filtres successivement, puis on relance la détection. Ce
qui reste n'est ni un VPN, ni une IP partagée : c'est un déplacement
impossible **réel** — un **compte compromis** (*account takeover*).

**Les deux filtres sont complémentaires :** le VPN mono-utilisateur échappe à
l'heuristique (exercice 4) mais pas à la liste (exercice 3). L'IP d'entreprise échappe à
la liste mais pas à l'heuristique. Il faut **les deux**.


<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

Complétez `utilisateurs_compromis` : elle renvoie la liste triée des utilisateurs au déplacement impossible réel, une fois les VPN et les IP partagées écartés.

</fieldset>

In [ ]:
def utilisateurs_compromis(df: pd.DataFrame, vpn_ips: pd.DataFrame) -> list[str]:
    """Utilisateurs au déplacement impossible réel, VPN et IP partagée écartés."""
    # BEGIN
    # raise NotImplementedError
    sans_vpn = retirer_vpn(df, vpn_ips)
    partagees = ip_partagees(sans_vpn)
    propres = sans_vpn[~sans_vpn["ip"].isin(partagees)]
    alertes = detecter_voyages_impossibles(propres)
    return sorted(alertes["username"].unique())
    # END



In [ ]:
def check_compromis():
    compromis = utilisateurs_compromis(df, vpn)
    assert compromis == ["alice.dupont"], f"trouvé : {compromis}"
    print("ex 5 OK : le compte compromis est", compromis[0])

check_compromis()


On regarde ici de plus près le déplacement du compte compromis.


In [ ]:
alerte = detecter_voyages_impossibles(df[df["username"] == "alice.dupont"])
print(alerte[["timestamp", "ip", "t_prev", "ip_prev",
             "distance_km", "vitesse_kmh"]].to_string(index=False))

## Bilan

| exercice | faux positif éliminé | méthode | notion pandas |
| --- | --- | --- | --- |
| 2 | (aucun — détection brute) | seuil de vitesse > 900 km/h | `sort_values` + boucle sur le précédent |
| 3 | VPN (5 télétravailleurs) | liste externe `vpn_ips.csv` | `isin` |
| 4 | IP d'entreprise (12 employés) | IP partagée par N utilisateurs | `groupby().nunique()` |
| 5 | — | il ne reste que `alice.dupont` | enchaînement des filtres |

**À retenir :**

- La **position d'une IP n'est pas la position de l'utilisateur** (VPN, NAT,
  proxy) : une géolocalisation se lit avec prudence.
- La détection d'*impossible travel* se **croise toujours** avec d'autres
  signaux (liste de VPN, IP partagées) pour écarter les faux positifs.
- Chaque faux positif éliminé rapproche du vrai signal.

